# test_model.ipynb

**Pipeline position:** Step 3 — runs *after* `key_authors_graph.ipynb`.

**Inputs (per campaign, from `key_authors_graph.ipynb` output):**
* `*_node_indicators.parquet` — author-level feature DataFrame, indexed by `accountid`.
* `processed_<campaign>_part_all.gzip.parquet` — co-located post-level file used to derive the IO label via the `is_control` column.

**Experiments:**
1. **Intra-campaign** — train/test on the same campaign, varying the training fraction.
2. **Inter-campaign (LOO)** — train on the other 5 campaigns, test on the held-out target.

**Outputs:** Results CSV, AUC plot, Macro-F1 plot.

# Imports

In [ ]:
from __future__ import annotations

# File system
from pathlib import Path
import os
import gc
import time

# Data
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt

# sklearn & models
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, f1_score
from sklearn.model_selection import train_test_split

# Utilities
from functools import wraps

# Parameters

> Papermill will inject values into the tagged cell below.  
> `target_campaign` is kept for single-campaign papermill runs; the sweep always covers all `CAMPAIGNS`.

In [ ]:
# --- Papermill parameters (override via -p flag) ---
target_campaign         = "Egypt_UAE"    # kept for single-run papermill compatibility
graph_filtering_setting = "all_authors"  # selects which node_indicators variant to load

# Configuration

In [ ]:
# Each campaign name must correspond to a folder under results/Labeled_Datasets/
# that contains the output of a completed key_authors_graph.ipynb run.
CAMPAIGNS = ["Egypt_UAE", "Cuba", "Russia", "Venezuela", "Iran", "China"]

# Training fractions swept from 0.1 % to 100 %
FRACTIONS = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 1.0]

MODELS = ["xgboost", "random_forest", "logistic_regression"]

RANDOM_STATE = 42
TEST_SIZE    = 0.20

# Utils

In [ ]:
BASE_DIR = Path(os.getenv("WORKSPACE_DIR", "/home/idanmu/Backbone-Graph"))

FOLDER_OUTPUT_PATH = BASE_DIR / "results" / "test_model"
FOLDER_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
REPORT_PATH = FOLDER_OUTPUT_PATH / "progress_report.txt"


def write_report(msg: str) -> None:
    """Log a timestamped message to stdout and the progress report file.

    Args:
        msg: Message string to log.
    """
    line = f"{time.strftime('%H:%M:%S')} | {msg}"
    print(line)
    with open(REPORT_PATH, "a") as fh:
        fh.write(line + "\n")


def report_done(func):
    """Decorator: log a completion entry after the wrapped function returns.

    Args:
        func: Function to wrap.

    Returns:
        Wrapped function.
    """
    @wraps(func)
    def wrapper(*args, **kwargs):
        try:
            result = func(*args, **kwargs)
            write_report(f"Completed: {func.__name__}")
            return result
        except Exception as exc:
            write_report(f"FAILED: {func.__name__} | {exc}")
            raise
    return wrapper


def get_classifier(model_name: str, random_state: int = RANDOM_STATE, **kwargs):
    """Return a sklearn-compatible classifier by name.

    Args:
        model_name: One of "xgboost"/"xgb", "random_forest"/"rf",
            "logistic_regression"/"lr".
        random_state: Random seed forwarded to the classifier constructor.
        **kwargs: Additional keyword arguments passed to the constructor.

    Returns:
        An untrained sklearn-compatible classifier instance.

    Raises:
        ValueError: If model_name is not recognised.
    """
    name = model_name.lower()
    if name in ("xgboost", "xgb"):
        return XGBClassifier(random_state=random_state, eval_metric="logloss", **kwargs)
    if name in ("random_forest", "rf"):
        return RandomForestClassifier(random_state=random_state, **kwargs)
    if name in ("logistic_regression", "lr"):
        return LogisticRegression(random_state=random_state, solver="liblinear", max_iter=1000, **kwargs)
    raise ValueError(
        f"Unsupported model_name: '{model_name}'. "
        "Supported: xgboost/xgb, random_forest/rf, logistic_regression/lr."
    )

# Data Loading

Each campaign is expected to have a completed `key_authors_graph.ipynb` run under
`results/Labeled_Datasets/{campaign_name}/`.  That run saves:

| File | Purpose |
|------|----------|
| `accounts_graph_topic_col_*_node_indicators.parquet` | Author-level feature DataFrame (no graph filtering) |
| `accounts_graph_top_percentage_*_node_indicators.parquet` | Filtered variants |
| `processed_{campaign}_part_all.gzip.parquet` | Post-level data; `is_control` column gives the IO labels |

The `graph_filtering_setting` parameter selects which node-indicators variant to load.

In [ ]:
def _match_indicator_file(fname: str, setting: str) -> bool:
    """Return True if the node-indicators filename matches the requested setting.

    Args:
        fname: Filename (not full path) of a candidate parquet file.
        setting: Graph-filtering setting string. "all_authors" selects the
            unfiltered variant (no "top_percentage" in the filename).
            Any other value is matched as a substring of the filename.

    Returns:
        True if the file matches the requested setting.
    """
    if "_node_indicators" not in fname:
        return False
    if setting == "all_authors":
        # Unfiltered: graph built on all authors, no top_percentage in name
        return "top_percentage" not in fname
    return setting in fname


def load_campaign_data(
    campaign_name: str,
    graph_filter: str = "all_authors",
    prefix_ids: bool = False,
) -> tuple[pd.DataFrame, set]:
    """Load the node-indicator DataFrame and IO author set for one campaign.

    Searches recursively under ``results/Labeled_Datasets/{campaign_name}/``
    for the node-indicators parquet produced by ``key_authors_graph.ipynb``
    and the co-located processed parquet that carries the ``is_control`` label.

    Args:
        campaign_name: Subfolder name under ``results/Labeled_Datasets/``.
            Must correspond to a completed ``key_authors_graph.ipynb`` run.
        graph_filter: Selects which node-indicators variant to load.
            ``"all_authors"`` loads the unfiltered (full-graph) file.
            Any other string is matched as a substring of the filename
            (e.g. ``"top_percentage_0.1_per_topic"`` selects that variant).
        prefix_ids: If ``True``, prepend ``"{campaign_name}_"`` to every
            ``accountid`` so that authors from different campaigns have
            collision-free identifiers when DataFrames are concatenated.

    Returns:
        df_features: Author-level DataFrame indexed by ``accountid``,
            numeric columns only, ready for classification.
        io_set: Set of ``accountid`` strings labelled as IO
            (``is_control == False`` in the processed parquet).

    Raises:
        FileNotFoundError: If the campaign directory, node-indicators file,
            or processed label file cannot be located.
    """
    campaign_name = campaign_name.strip()
    camp_dir = BASE_DIR / "results" / "Labeled_Datasets" / campaign_name

    if not camp_dir.exists():
        raise FileNotFoundError(
            f"Campaign directory not found: {camp_dir}\n"
            "Has key_authors_graph.ipynb been run for this campaign?"
        )

    # ── Locate node_indicators file ──────────────────────────────────────────
    candidate_files = [
        f for f in camp_dir.rglob("*_node_indicators.parquet")
        if _match_indicator_file(f.name, graph_filter)
    ]
    if not candidate_files:
        raise FileNotFoundError(
            f"No node_indicators file found for '{campaign_name}' "
            f"with graph_filtering_setting='{graph_filter}'.\n"
            f"Searched under: {camp_dir}"
        )
    # When multiple dates exist, take the most recent (lexicographic sort on full path)
    df_features = pd.read_parquet(sorted(candidate_files)[-1])

    # ── Locate processed parquet for IO labels ───────────────────────────────
    processed_candidates = list(
        camp_dir.rglob(f"processed_{campaign_name}_part_all.gzip.parquet")
    )
    if not processed_candidates:
        raise FileNotFoundError(
            f"No processed parquet found for campaign '{campaign_name}' "
            f"under {camp_dir}"
        )
    df_proc = pd.read_parquet(
        sorted(processed_candidates)[-1],
        columns=["accountid", "is_control"],
    )
    io_set = set(
        df_proc.loc[df_proc["is_control"] == False, "accountid"].astype(str)
    )

    if prefix_ids:
        df_features.index = campaign_name + "_" + df_features.index.astype(str)
        io_set = {f"{campaign_name}_{aid}" for aid in io_set}

    n_io_in_graph = len(set(df_features.index.astype(str)) & io_set)
    write_report(
        f"Loaded '{campaign_name}' | {len(df_features)} authors in graph "
        f"| {len(io_set)} IO authors (dataset) | {n_io_in_graph} IO in graph"
    )
    return df_features, io_set


def load_multiple_campaigns(
    campaign_list: list[str],
    graph_filter: str = "all_authors",
) -> tuple[pd.DataFrame, set]:
    """Concatenate node-indicator DataFrames from multiple campaigns.

    All author IDs are prefixed with their campaign name before concatenation
    to prevent cross-campaign accountid collisions.

    Args:
        campaign_list: Ordered list of campaign names to load and combine.
        graph_filter: Graph-filtering setting forwarded to
            :func:`load_campaign_data` for each campaign.

    Returns:
        Combined author-level DataFrame (index prefixed with campaign name)
        and the union of all IO author sets.
    """
    frames: list[pd.DataFrame] = []
    combined_io: set = set()

    for camp in campaign_list:
        try:
            df, io_set = load_campaign_data(
                camp, graph_filter=graph_filter, prefix_ids=True
            )
            frames.append(df)
            combined_io.update(io_set)
        except FileNotFoundError as exc:
            write_report(f"WARNING: Skipping '{camp}' in multi-campaign load — {exc}")

    if not frames:
        raise FileNotFoundError(
            f"None of the requested campaigns could be loaded: {campaign_list}"
        )

    return pd.concat(frames, axis=0), combined_io

# Classification Engine

Mirrors `io_classification` from `key_authors_graph.ipynb` exactly, with an
additional cross-campaign test path so that the same function handles both
intra-campaign (internal train/test split) and inter-campaign (external test
set) experiments.

In [ ]:
@report_done
def io_classification(
    node_indicators_df: pd.DataFrame,
    io_authors_set: set,
    model_name: str,
    test_size: float = TEST_SIZE,
    random_state: int = RANDOM_STATE,
    test_node_indicators_df: pd.DataFrame | None = None,
    test_io_authors_set: set | None = None,
) -> tuple:
    """Train and evaluate an IO classifier.

    Classification logic mirrors ``key_authors_graph.ipynb`` exactly.

    Two modes:

    * **Intra-campaign** (default) — performs ``train_test_split`` on
      *node_indicators_df*.
    * **Cross-campaign** — bypasses the internal split and uses the explicitly
      supplied *test_node_indicators_df* / *test_io_authors_set*. Feature
      columns must match exactly between train and test DataFrames.

    Args:
        node_indicators_df: Training-set author DataFrame, indexed by
            ``accountid``, numeric columns are used as features.
        io_authors_set: Set of IO ``accountid`` strings for the training data.
        model_name: Model identifier — see :func:`get_classifier`.
        test_size: Fraction held out for testing in intra-campaign mode.
        random_state: Random seed used for splitting and the classifier.
        test_node_indicators_df: External test DataFrame for cross-campaign
            mode.  Must have the same numeric columns as the training frame.
        test_io_authors_set: IO author set matching *test_node_indicators_df*.

    Returns:
        tuple(clf, report, auc, macro_f1):
            clf       – Trained classifier instance.
            report    – Classification report string.
            auc       – ROC-AUC score (0.5 when only one class is present).
            macro_f1  – Macro-averaged F1 score.

    Raises:
        ValueError: If the training set is empty or train/test feature columns
            do not match in cross-campaign mode.
    """
    if node_indicators_df.empty:
        raise ValueError("Training feature set is empty — aborting.")

    # Only numeric columns are used as model features
    train_numeric = node_indicators_df.select_dtypes(include="number")
    train_cols    = train_numeric.columns.tolist()
    X_train       = train_numeric.to_numpy()

    io_str  = set(map(str, io_authors_set))
    y_train = node_indicators_df.index.astype(str).isin(io_str).astype(int)

    # ── Cross-campaign mode ──────────────────────────────────────────────────
    if test_node_indicators_df is not None and test_io_authors_set is not None:
        write_report("Cross-campaign mode: using external test set.")

        test_numeric = test_node_indicators_df.select_dtypes(include="number")
        test_cols    = test_numeric.columns.tolist()

        # Strict alignment: fail fast if feature sets diverge
        if train_cols != test_cols:
            mismatch = set(train_cols) ^ set(test_cols)
            raise ValueError(
                f"FATAL ALIGNMENT ERROR: Train/test feature columns do not match.\n"
                f"Train cols: {len(train_cols)}  |  Test cols: {len(test_cols)}\n"
                f"Mismatched: {mismatch}"
            )

        X_test       = test_numeric.to_numpy()
        test_io_str  = set(map(str, test_io_authors_set))
        y_test       = test_node_indicators_df.index.astype(str).isin(test_io_str).astype(int)

    # ── Intra-campaign mode ──────────────────────────────────────────────────
    else:
        stratify = y_train if len(np.unique(y_train)) > 1 else None
        X_train, X_test, y_train, y_test = train_test_split(
            X_train, y_train,
            test_size=test_size,
            random_state=random_state,
            stratify=stratify,
        )

    write_report(
        f"  [{model_name}] Train: {len(X_train)} | Test: {len(X_test)} "
        f"| IO in test: {int(y_test.sum())}"
    )

    clf = get_classifier(model_name=model_name, random_state=random_state)
    clf.fit(X_train, y_train)

    y_pred  = clf.predict(X_test)
    y_proba = (
        clf.predict_proba(X_test)[:, 1]
        if hasattr(clf, "predict_proba")
        else y_pred.astype(float)
    )

    report   = classification_report(
        y_test, y_pred, target_names=["Non-IO", "IO"], zero_division=0
    )
    auc      = roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) > 1 else 0.5
    macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

    return clf, report, auc, macro_f1

# Experiment Sweep

For every target campaign the sweep:
1. Loads the full node-indicator DataFrame.
2. Creates a **fixed 80 / 20 train/test split** (same seed → identical test set
   across all fractions and models).
3. Verifies **zero overlap** between the sampled train set and the test set.
4. Runs **intra-campaign** experiments (train on current campaign).
5. Runs **inter-campaign / LOO** experiments (train on combined other 5
   campaigns, test on the same held-out 20 %).

In [ ]:
def _stratified_sample(
    df: pd.DataFrame,
    io_str_set: set,
    n: int,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    """Sample n rows from df, guaranteeing at least 1 IO and 1 non-IO author.

    When only one class exists in the pool (e.g. no IO authors reached the
    graph), falls back to a plain random sample so the caller can catch the
    single-class situation through the normal error path.

    Args:
        df: Author-level DataFrame to sample from.
        io_str_set: Set of IO accountid strings (already prefixed if needed).
        n: Desired sample size (≥ 2).
        random_state: Random seed.

    Returns:
        Sampled DataFrame with at most *n* rows, both classes represented.
    """
    y         = df.index.astype(str).isin(io_str_set).astype(int)
    io_idx    = df.index[y == 1].tolist()
    nonIO_idx = df.index[y == 0].tolist()

    # If only one class available, plain sample — caller handles the edge case
    if not io_idx or not nonIO_idx:
        return df.sample(n=min(n, len(df)), random_state=random_state)

    # Proportional allocation, minimum 1 of each class
    prop_io = len(io_idx) / len(df)
    n_io    = max(1, min(int(round(n * prop_io)), len(io_idx), n - 1))
    n_nonIO = max(1, min(n - n_io, len(nonIO_idx)))

    rng           = np.random.default_rng(random_state)
    sampled_io    = rng.choice(io_idx,    size=n_io,    replace=False).tolist()
    sampled_nonIO = rng.choice(nonIO_idx, size=n_nonIO, replace=False).tolist()

    return df.loc[sampled_io + sampled_nonIO]


def run_experiment_sweep(
    campaigns: list[str],
    fractions: list[float],
    models: list[str],
    graph_filter: str = "all_authors",
) -> pd.DataFrame:
    """Run the full intra / LOO inter-campaign classification sweep.

    For every target campaign the function:

    1. Loads the node-indicator DataFrame and IO set.
    2. Creates a **fixed 80 / 20 train/test split** (same seed → identical
       test set across all fractions and models).
    3. For each fraction, draws a **stratified sample** from the 80 % train
       pool so that both classes are always represented, even at 0.1 %.
    4. Verifies **zero overlap** between the sampled train set and the test set.
    5. Runs **intra-campaign** and **LOO inter-campaign** experiments.

    Args:
        campaigns: Campaign names — each must have a completed pipeline run
            under ``results/Labeled_Datasets/``.
        fractions: Training fractions to sweep, values in (0, 1].
        models: Model name strings passed to :func:`get_classifier`.
        graph_filter: Graph-filtering setting forwarded to
            :func:`load_campaign_data`.

    Returns:
        DataFrame with columns:
        ``training_campaign``, ``target_campaign``,
        ``graph_filtering_setting``, ``train_fraction``,
        ``model``, ``experiment_type``, ``auc``, ``macro_f1``.
    """
    results: list[dict] = []

    for target_camp in campaigns:
        write_report("=" * 60)
        write_report(f"Target campaign: {target_camp}")

        # ── Load target campaign ─────────────────────────────────────────────
        try:
            df_target, io_target = load_campaign_data(
                target_camp, graph_filter=graph_filter, prefix_ids=False
            )
        except FileNotFoundError as exc:
            write_report(f"SKIP {target_camp}: {exc}")
            continue

        io_target_str = set(map(str, io_target))
        y_all = df_target.index.astype(str).isin(io_target_str).astype(int)

        if y_all.sum() == 0:
            write_report(
                f"SKIP {target_camp}: no IO authors in the graph — "
                "cannot produce a meaningful evaluation."
            )
            continue

        # Fixed 80/20 split — test set is identical for all fractions and models
        stratify = y_all if len(np.unique(y_all)) > 1 else None
        df_train_full, df_test, y_train_full, y_test_ref = train_test_split(
            df_target, y_all,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
            stratify=stratify,
        )
        test_io_set = set(df_test.index[y_test_ref == 1].astype(str))

        write_report(
            f"  Train pool: {len(df_train_full)} | Test: {len(df_test)} "
            f"| IO in test: {int(y_test_ref.sum())}"
        )

        # ── LOO: load other 5 campaigns ──────────────────────────────────────
        loo_camps = [c for c in campaigns if c != target_camp]
        try:
            df_inter_all, io_inter_all = load_multiple_campaigns(
                loo_camps, graph_filter=graph_filter
            )
            write_report(
                f"  LOO train pool: {len(df_inter_all)} authors from {loo_camps}"
            )
        except FileNotFoundError as exc:
            write_report(f"  WARNING: LOO data unavailable — {exc}")
            df_inter_all, io_inter_all = None, None

        # ── Fraction × model sweep ────────────────────────────────────────────
        for frac in fractions:
            # ── Intra: stratified sample from 80 % train pool ────────────────
            n_intra = max(2, min(int(round(len(df_train_full) * frac)), len(df_train_full)))

            if frac >= 1.0:
                df_train_frac = df_train_full
            else:
                # Stratified: guarantee ≥1 IO + ≥1 non-IO in every sample
                df_train_frac = _stratified_sample(
                    df_train_full, io_target_str, n_intra, RANDOM_STATE
                )

            # Strict no-overlap check (guaranteed: test set was carved out first)
            overlap = set(df_train_frac.index.astype(str)) & set(df_test.index.astype(str))
            assert len(overlap) == 0, (
                f"FATAL: {len(overlap)} authors appear in both train and test sets "
                f"for '{target_camp}' at fraction {frac}!"
            )

            train_io_frac = set(df_train_frac.index.astype(str)) & io_target_str
            write_report(
                f"  frac={frac*100:.1f}% | train_n={len(df_train_frac)} "
                f"| IO_in_train={len(train_io_frac)}"
            )

            for model_name in models:
                # ─ Intra ────────────────────────────────────────────────────
                try:
                    _, _, auc, macro_f1 = io_classification(
                        node_indicators_df=df_train_frac,
                        io_authors_set=train_io_frac,
                        model_name=model_name,
                        test_node_indicators_df=df_test,
                        test_io_authors_set=test_io_set,
                    )
                    results.append({
                        "training_campaign":       target_camp,
                        "target_campaign":         target_camp,
                        "graph_filtering_setting": graph_filter,
                        "train_fraction":          frac,
                        "model":                   model_name,
                        "experiment_type":         "intra",
                        "auc":                     auc,
                        "macro_f1":                macro_f1,
                    })
                except Exception as exc:
                    write_report(
                        f"  ERROR intra | {target_camp} | {model_name} | frac={frac}: {exc}"
                    )

                # ─ Inter (LOO) ───────────────────────────────────────────────
                if df_inter_all is None:
                    continue

                try:
                    n_inter = max(2, min(int(round(len(df_inter_all) * frac)), len(df_inter_all)))

                    if frac >= 1.0:
                        df_inter_frac = df_inter_all
                    else:
                        df_inter_frac = _stratified_sample(
                            df_inter_all, set(map(str, io_inter_all)),
                            n_inter, RANDOM_STATE
                        )

                    inter_io_frac = (
                        set(df_inter_frac.index.astype(str))
                        & set(map(str, io_inter_all))
                    )

                    _, _, auc, macro_f1 = io_classification(
                        node_indicators_df=df_inter_frac,
                        io_authors_set=inter_io_frac,
                        model_name=model_name,
                        test_node_indicators_df=df_test,
                        test_io_authors_set=test_io_set,
                    )
                    results.append({
                        "training_campaign":       ",".join(loo_camps),
                        "target_campaign":         target_camp,
                        "graph_filtering_setting": graph_filter,
                        "train_fraction":          frac,
                        "model":                   model_name,
                        "experiment_type":         "inter",
                        "auc":                     auc,
                        "macro_f1":                macro_f1,
                    })
                except Exception as exc:
                    write_report(
                        f"  ERROR inter | {target_camp} | {model_name} | frac={frac}: {exc}"
                    )

        # Free memory before next campaign
        del df_target, df_train_full
        if df_inter_all is not None:
            del df_inter_all
        gc.collect()

    return pd.DataFrame(results)

# Execution

In [ ]:
write_report("Starting cross-campaign IO detection sweep.")
write_report(f"Campaigns  : {CAMPAIGNS}")
write_report(f"Fractions  : {FRACTIONS}")
write_report(f"Models     : {MODELS}")
write_report(f"Graph filter: {graph_filtering_setting}")
write_report(f"Output dir : {FOLDER_OUTPUT_PATH}")

results_df = run_experiment_sweep(
    campaigns=CAMPAIGNS,
    fractions=FRACTIONS,
    models=MODELS,
    graph_filter=graph_filtering_setting,
)

if results_df.empty:
    write_report(
        "WARNING: No results collected. "
        "Ensure all campaigns have completed key_authors_graph.ipynb runs."
    )
else:
    write_report(f"Sweep complete — {len(results_df)} result rows collected.")
    display(results_df.head(12))

# Save Results

In [ ]:
if not results_df.empty:
    csv_path = FOLDER_OUTPUT_PATH / f"experiment_results_{graph_filtering_setting}.csv"
    # Keep only the required output columns (plus model + experiment_type for traceability)
    output_cols = [
        "training_campaign",
        "target_campaign",
        "graph_filtering_setting",
        "train_fraction",
        "model",
        "experiment_type",
        "auc",
        "macro_f1",
    ]
    results_df[output_cols].to_csv(csv_path, index=False)
    write_report(f"Results saved → {csv_path}")
    display(results_df[output_cols].describe())

# Plots

Two figures are produced — one for **AUC** and one for **Macro-F1**:

* **2 × 3 grid**, one subplot per campaign.
* **Orange/gold** line — intra-campaign (IOHunter); **blue** line — inter-campaign LOO (Node2Vec).
* **Error bars** = std across the three models at each fraction.
* **Log-scale X-axis**: 0.1 % → 100 %.
* **Zoomed inset** on the low-fraction region (0.1 %–10 %).

In [ ]:
LINES = [
    ("intra", "IOHunter", "#D4A017", "o"),   # gold/orange, circle
    ("inter", "Node2Vec", "#4472C4", "s"),    # blue, square
]

CAMPAIGN_ORDER = ["Egypt_UAE", "Cuba", "Russia", "Venezuela", "Iran", "China"]


def plot_cross_campaign_performance(
    df_results: pd.DataFrame,
    metric: str,
    output_dir: Path | None = None,
) -> None:
    """Plot intra- vs inter-campaign performance with a zoomed low-fraction inset.

    Produces a 2 × 3 subplot grid (one subplot per campaign).  Each subplot
    shows two lines — IOHunter (intra, orange) and Node2Vec (inter, blue) —
    with ±1 std error bars and a zoomed inset in the upper-left corner
    covering the 0.1 %–10 % low-fraction region.

    Args:
        df_results: Output of the experiment sweep.  Must contain columns
            ``target_campaign``, ``experiment_type``, ``train_fraction``,
            and the requested *metric* column.
        metric: Column to visualise — ``"auc"`` or ``"macro_f1"``.
        output_dir: Directory to save the PNG at 300 dpi.  Not saved if
            ``None``.
    """
    campaigns = [c for c in CAMPAIGN_ORDER if c in df_results["target_campaign"].unique()]
    if not campaigns:
        write_report(f"No data to plot for metric '{metric}'.")
        return

    fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 10))
    axes_flat = axes.flatten()

    for idx, camp in enumerate(campaigns):
        ax = axes_flat[idx]
        camp_data = df_results[df_results["target_campaign"] == camp]

        # ── Main lines (one per experiment type) ──────────────────────────────
        for exp_type, label, color, marker in LINES:
            subset = camp_data[camp_data["experiment_type"] == exp_type]
            if subset.empty:
                continue
            grouped = (
                subset.groupby("train_fraction")[metric]
                .agg(["mean", "std"])
                .reset_index()
            )
            ax.errorbar(
                grouped["train_fraction"] * 100,
                grouped["mean"],
                yerr=grouped["std"].fillna(0),
                fmt=f"-{marker}",
                color=color,
                label=label,
                capsize=3,
                linewidth=1.5,
                markersize=6,
                elinewidth=1.0,
            )

        # ── X-axis: log scale, fixed readable ticks ───────────────────────────
        ax.set_xscale("log")
        ax.set_xticks([0.1, 1, 5, 10, 25, 50, 100])
        ax.set_xticklabels([".1", "1", "5", "10", "25", "50", "100"])
        ax.set_xlim(0.08, 110)

        # ── Per-subplot labels and legend ─────────────────────────────────────
        ax.set_title(camp, fontweight="bold", fontsize=11)
        ax.set_xlabel("Percentage of Training Data (%)", fontsize=9)
        ax.set_ylabel(metric.upper(), fontsize=9)
        ax.legend(fontsize=8, loc="lower right")
        ax.grid(True, linestyle="--", alpha=0.4)
        ax.set_ylim(bottom=0)

        # ── Inset zoom: upper-left, 0.1 %–10 % region only ───────────────────
        axins = ax.inset_axes([0.03, 0.45, 0.42, 0.50])

        for exp_type, label, color, marker in LINES:
            subset = camp_data[camp_data["experiment_type"] == exp_type]
            if subset.empty:
                continue
            grouped = (
                subset.groupby("train_fraction")[metric]
                .agg(["mean", "std"])
                .reset_index()
            )
            low = grouped[grouped["train_fraction"] <= 0.10]   # 0.1 % – 10 %
            if low.empty:
                continue
            axins.errorbar(
                low["train_fraction"] * 100,
                low["mean"],
                yerr=low["std"].fillna(0),
                fmt=f"-{marker}",
                color=color,
                capsize=3,
                linewidth=1.2,
                markersize=4,
            )

        axins.set_xscale("log")
        axins.set_xticks([0.1, 1.0, 5.0, 10.0])
        axins.set_xticklabels([".1", "1", "5", "10"], fontsize=7)
        axins.tick_params(axis="y", labelsize=7)
        axins.grid(True, linestyle="--", alpha=0.4)
        axins.set_xlim(0.08, 12)
        axins.set_ylim(bottom=0)   # Bug 1 fix: prevent error bars pulling y below zero

        # Clean black border — no indicate_inset_zoom (avoids diagonal connectors)
        for spine in axins.spines.values():
            spine.set_edgecolor("black")
            spine.set_linewidth(1.2)

        # Bug 2 fix: derive rect bounds from the inset's actual y-range so the
        # dashed rectangle on the main plot matches exactly what the inset shows,
        # regardless of the metric's value distribution.
        ax.axvspan(0.08, 12, alpha=0.04, color="grey")
        rect_x = [0.08, 12, 12, 0.08, 0.08]
        rect_y_min, rect_y_max = axins.get_ylim()
        ax.plot(
            rect_x,
            [rect_y_min, rect_y_min, rect_y_max, rect_y_max, rect_y_min],
            "grey",
            linewidth=0.8,
            linestyle="--",
        )

    # ── Hide unused subplot slots ─────────────────────────────────────────────
    for idx in range(len(campaigns), len(axes_flat)):
        axes_flat[idx].set_visible(False)

    # ── Figure-level title, layout, and save ──────────────────────────────────
    fig.suptitle(
        f"Intra- vs Inter-Campaign IO Detection — {metric.upper()}",
        fontsize=14,
        fontweight="bold",
    )
    plt.tight_layout()
    if output_dir:
        out_path = output_dir / f"strategy_comparison_{metric}.png"
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        write_report(f"Saved plot → {out_path}")
    plt.show()

In [ ]:
if not results_df.empty:
    for metric in ["auc", "macro_f1"]:
        plot_cross_campaign_performance(
            df_results=results_df,
            metric=metric,
            output_dir=FOLDER_OUTPUT_PATH,
        )

write_report("Pipeline completed.")